<a href="https://colab.research.google.com/github/kandinz/Omni-TTS/blob/main/OmmiVoice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiên Đoàn TTS — Giao Diện Tạo Audio & Batch JSON

> **GPU:** T4 16GB (tự động) · **Chế độ:** UI Trực quan & Batch Generation từ JSON · **Tốc độ:** Build 1 lần (~3 phút), các lần sau thay đổi text xuất audio tức thì (~3-5s/cảnh)

## 🚀 Hướng dẫn sử dụng

1. **Bước 1 & 2:** Nhấn **Runtime → Run all (Ctrl+F9)** hoặc chạy Cell 1 & Cell 2 để nạp thư viện và Model (**chỉ cần Build 1 lần**).
2. **Bước 3:** Sử dụng **Giao diện Web UI (Gradio)** để nhập Text hoặc kịch bản JSON. Thay đổi văn bản và bấm **Tạo giọng nói** để xuất audio ngay lập tức mà không cần load lại Model!

In [1]:
# # 1. Kết nối Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# # 2. Cài đặt huggingface-hub
# !pip install -q huggingface_hub
# # 3. Tải Model về Drive (Sửa lại tên model và đường dẫn thư mục tùy ý)
# # Ví dụ tải model "k2-fsa/OmniVoice" về thư mục "HuggingFace_Models" trên Drive
# !huggingface-cli download k2-fsa/OmniVoice --local-dir /content/drive/MyDrive/HuggingFace_Models/OmniVoice --local-dir-use-symlinks False

Mounted at /content/drive

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.23.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [2]:
# === BƯỚC 1: CÀI ĐẶT THƯ VIỆN, KẾT NỐI DRIVE & TẢI VOICE SAMPLE ===
import os

print('🚀 [1/3] Đang kết nối Google Drive và kiểm tra môi trường...')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Đặt thư mục cache của HuggingFace vào Drive để tải thẳng model về đây
    HF_CACHE_DIR = '/content/drive/MyDrive/OmniVoice_Cache/huggingface'
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = HF_CACHE_DIR
    print(f'✅ Đã cấu hình lưu Cache Model tại: {HF_CACHE_DIR}')
except Exception as e:
    print('⚠️ Không thể kết nối Google Drive (có thể không chạy trên Colab). Cache sẽ lưu mặc định.')

try:
    import omnivoice
    import gradio
    print('✅ Thư viện OmniVoice & Gradio đã được cài đặt sẵn!')
except ImportError:
    print('📦 Đang cài đặt thư viện cần thiết (~1-2 phút)...')
    !pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4" scipy tqdm
    !pip uninstall -y transformers
    !pip install -q "transformers>=5.3.0"
    print('✅ Cài đặt thư viện hoàn tất!')

VOICE_SAMPLE_FILE = "voice_sample.mp3"
VOICE_SAMPLE_URL = "https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/voice-sample/MinhQuanPro.wav"

if not os.path.exists(VOICE_SAMPLE_FILE):
    print(f'📥 Đang tải voice sample mẫu ({VOICE_SAMPLE_FILE})...')
    !wget -q "{VOICE_SAMPLE_URL}" -O "{VOICE_SAMPLE_FILE}"
    print(f'✅ Đã tải thành công {VOICE_SAMPLE_FILE}!')
else:
    print(f'✅ File {VOICE_SAMPLE_FILE} đã sẵn sàng!')


🚀 [1/3] Đang kết nối Google Drive và kiểm tra môi trường...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã cấu hình lưu Cache Model tại: /content/drive/MyDrive/OmniVoice_Cache/huggingface
📦 Đang cài đặt thư viện cần thiết (~1-2 phút)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.5 MB/s eta 0:00:00
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 61.5 MB/s eta 0:00:00
✅ Cài đặt thư viện hoàn tất!
📥 Đang tải voice sample mẫu (voice_sample.mp3)...
✅ Đã tải thành công voice_sample.mp3!


In [ ]:
# === BƯỚC 2: KHỞI TẠO OMNIVOICE MODEL & VOICE PROMPT (BUILD 1 LẦN) ===
print('🤖 [2/3] Đang khởi động OmniVoice Model...')

import logging, time, os, re, json
import numpy as np
import torch
import scipy.io.wavfile as wavfile

# Shim: AutoFeatureExtractor removed in transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Kiểm tra GPU CUDA
if torch.cuda.is_available():
    print(f'⚡ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ CẢNH BÁO: Không tìm thấy GPU! Hãy chọn Runtime > Change runtime type > T4 GPU.')

# Nạp Model 1 lần duy nhất vào memory (globals)
DEVICE = get_best_device()
if 'model' not in globals():
    print(f'🧠 Đang nạp model OmniVoice vào {DEVICE} (float16)...')
    model = OmniVoice.from_pretrained(
        'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
    )
    SAMPLING_RATE = model.sampling_rate
    print(f'✅ Model ready — Sampling Rate: {SAMPLING_RATE}Hz')
else:
    print('✅ Model đã được load sẵn trong memory từ trước!')

# Nạp Voice Clone Prompt 1 lần duy nhất
if 'VOICE_PROMPT' not in globals() or VOICE_PROMPT is None:
    print(f'🎙️ Đang khởi tạo Voice Clone Prompt từ {VOICE_SAMPLE_FILE}...')
    VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio=VOICE_SAMPLE_FILE)
    print('✅ Voice Prompt ready!')
else:
    print('✅ Voice Prompt đã sẵn sàng!')

# Cấu hình sinh mặc định
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)


🤖 [2/3] Đang khởi động OmniVoice Model...


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


⚠️ CẢNH BÁO: Không tìm thấy GPU! Hãy chọn Runtime > Change runtime type > T4 GPU.
🧠 Đang nạp model OmniVoice vào cpu (float16)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
# === BƯỚC 3: GIAO DIỆN WEB UI (GRADIO) — TẠO AUDIO TỨC THÌ ===
# Model đã được build từ Bước 2, tại đây chỉ cần thay đổi text và bấm Tạo để xuất audio.

import gradio as gr
import shutil
import json
import os
import time
import re
import numpy as np
import scipy.io.wavfile as wavfile
import torch
import base64

# Hàm tạo giọng nói từ 1 đoạn văn bản
def generate_single_text(text: str, speed: float):
    text = text.strip()
    if not text:
        return None, "⚠️ Vui lòng nhập văn bản cần tạo giọng nói."

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None, "⚠️ Văn bản không hợp lệ."

    start_t = time.time()
    with torch.inference_mode():
        if len(paragraphs) == 1:
            audio = model.generate(
                text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=speed, generation_config=GEN_CFG
            )[0]
        else:
            audios = []
            for i, p in enumerate(paragraphs):
                a = model.generate(
                    text=p, voice_clone_prompt=VOICE_PROMPT,
                    language='vi', speed=speed, generation_config=GEN_CFG
                )[0]
                audios.append(a)
                if i < len(paragraphs) - 1:
                    audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
            audio = np.concatenate(audios)

    waveform = (audio * 32767).astype(np.int16)
    duration = len(waveform) / SAMPLING_RATE
    gen_time = time.time() - start_t

    status_msg = f"✅ Đã tạo audio thành công! Độ dài: {duration:.1f}s | Thời gian xử lý: {gen_time:.2f}s"
    return (SAMPLING_RATE, waveform), status_msg

# Hàm xử lý chung cho cả 2 nút
def generate_batch_json_core(json_str: str, speed: float, create_zip: bool):
    try:
        scenes = json.loads(json_str)
    except Exception as e:
        return f"❌ Lỗi cú pháp JSON: {e}", None, None, ""

    if not isinstance(scenes, list):
        return "❌ Dữ liệu JSON phải là danh sách (Array)", None, None, ""

    output_dir = "output_audio"
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    start_t = time.time()
    generated_count = 0
    generated_files = []

    with torch.inference_mode():
        for idx, item in enumerate(scenes):
            fname = item.get('filename', f'scene_{idx+1:02d}.wav')
            text = item.get('text', '').strip()
            if not text:
                continue

            if not fname.endswith('.wav'):
                fname += '.wav'

            paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
            if len(paragraphs) == 1:
                audio = model.generate(
                    text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
                    language='vi', speed=speed, generation_config=GEN_CFG
                )[0]
            else:
                audios = []
                for i, p in enumerate(paragraphs):
                    a = model.generate(
                        text=p, voice_clone_prompt=VOICE_PROMPT,
                        language='vi', speed=speed, generation_config=GEN_CFG
                    )[0]
                    audios.append(a)
                    if i < len(paragraphs) - 1:
                        audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
                audio = np.concatenate(audios)

            waveform = (audio * 32767).astype(np.int16)
            save_path = os.path.join(output_dir, fname)
            wavfile.write(save_path, SAMPLING_RATE, waveform)
            generated_files.append(save_path)
            generated_count += 1

    zip_filename = None
    if create_zip and generated_files:
        zip_filename = "output_audio.zip"
        base_zip = os.path.splitext(zip_filename)[0]
        shutil.make_archive(base_zip, "zip", output_dir)

    # Generate HTML for audio previews
    audio_html = "<div style='display: flex; flex-direction: column; gap: 10px;'>"
    for f in generated_files:
        try:
            with open(f, "rb") as audio_file:
                b64 = base64.b64encode(audio_file.read()).decode("utf-8")
            filename = os.path.basename(f)
            audio_html += f"<div><span style='font-weight: 500; font-size: 14px; margin-bottom: 4px; display: block;'>{filename}</span><audio controls src='data:audio/wav;base64,{b64}' style='width: 100%; height: 32px; border-radius: 4px;'></audio></div>"
        except Exception as e:
            pass
    audio_html += "</div>"

    elapsed = time.time() - start_t
    status_msg = f"🎉 Hoàn tất sinh {generated_count}/{len(scenes)} file WAV trong {elapsed:.1f}s!"
    if create_zip:
        status_msg += " Đã đóng gói file ZIP."

    return status_msg, generated_files, zip_filename, audio_html

def generate_only(json_str: str, speed: float):
    return generate_batch_json_core(json_str, speed, create_zip=False)

def generate_and_zip(json_str: str, speed: float):
    return generate_batch_json_core(json_str, speed, create_zip=True)

# Kịch bản JSON mẫu mặc định
DEFAULT_JSON_TEXT = json.dumps([
  {
    "filename": "scene_01_intro.wav",
    "text": "4 thứ cơ bản để tạo ra một AI Agent đúng nghĩa. Vì nếu thiếu những thứ này, rất nhiều con Agent thực ra chỉ giống một chatbot được đặt tên cho hay hơn thôi. 4 thứ đó là MCP, Skill, Hook và Schedule."
  },
  {
    "filename": "scene_02_mcp_analogy.wav",
    "text": "Đầu tiên là MCP. Bạn cứ hình dung AI Agent giống như một trợ lý rất thông minh nhưng đang ngồi trong một căn phòng kín. Nếu bạn không đưa cho nó điện thoại, máy tính, tài khoản, công cụ... thì nó chỉ có thể ngồi đó trả lời câu hỏi, nó không thể thật sự làm việc với thế giới bên ngoài."
  }
], ensure_ascii=False, indent=2)

print('Khởi động Giao diện Web UI...')
gr.close_all()

CSS = ".gradio-container{max-width:900px!important;margin:0 auto!important;padding:16px!important}footer{display:none!important}"
THEME = gr.themes.Soft(primary_hue='indigo')

JS_AUTO_DOWNLOAD = '''
function(status, files, zip_file, html) {
    if (zip_file && zip_file.path) {
        const link = document.createElement('a');
        link.href = zip_file.url;
        link.download = zip_file.orig_name || 'output_audio.zip';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }
    return [status, files, zip_file, html];
}
'''

with gr.Blocks(title='Kiên Đoàn TTS — Voice Generator', theme=THEME, css=CSS) as demo:
    gr.Markdown(
        "# 🎙️ Kiên Đoàn TTS — AI Voice Generator\n"
        "**Giọng Nam Công Nghệ** · Model được lưu trong Drive (nếu có). Nhập văn bản và bấm tạo để xuất Audio ngay lập tức!"
    )

    with gr.Tabs():
        with gr.TabItem("📝 Tạo Audio từ Text"):
            text_input = gr.Textbox(
                label="Nhập văn bản",
                lines=5,
                placeholder="Nhập đoạn văn bản bạn muốn chuyển thành giọng nói tại đây...",
                value="Chào mừng bạn đến với hệ thống tạo giọng nói AI Kiên Đoàn TTS!"
            )
            speed_single = gr.Slider(minimum=0.7, maximum=1.5, value=0.95, step=0.05, label="Tốc độ đọc (Speed)")
            btn_single = gr.Button("🚀 Tạo Giọng Nói", variant="primary")

            audio_output = gr.Audio(label="Kết quả Audio phát trực tiếp")
            status_single = gr.Textbox(label="Trạng thái", interactive=False)

            btn_single.click(
                generate_single_text,
                inputs=[text_input, speed_single],
                outputs=[audio_output, status_single]
            )

        with gr.TabItem("📦 Tạo Audio Hàng Loạt từ JSON"):
            json_input = gr.Textbox(
                label="Kịch bản JSON (Array)",
                lines=10,
                placeholder="Nhập mảng JSON danh sách các cảnh cần sinh audio...",
                value=DEFAULT_JSON_TEXT
            )
            speed_batch = gr.Slider(minimum=0.7, maximum=1.5, value=0.95, step=0.05, label="Tốc độ đọc (Speed)")

            with gr.Row():
                btn_only_audio = gr.Button("▶️ Chỉ Tạo Audio", variant="secondary")
                btn_batch_zip = gr.Button("⚡ Tạo Audio & Tải ZIP", variant="primary")

            status_batch = gr.Textbox(label="Trạng thái xử lý", interactive=False)

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 🎧 Nghe trước")
                    preview_html = gr.HTML(label="Nghe trước")
                with gr.Column(scale=1):
                    gr.Markdown("### 📥 File tải về")
                    output_files = gr.File(label="Tải từng file (WAV)", file_count="multiple")
                    zip_output = gr.File(label="Tải toàn bộ (ZIP)")

            btn_only_audio.click(
                generate_only,
                inputs=[json_input, speed_batch],
                outputs=[status_batch, output_files, zip_output, preview_html]
            )

            btn_batch_zip.click(
                generate_and_zip,
                inputs=[json_input, speed_batch],
                outputs=[status_batch, output_files, zip_output, preview_html],
                js=JS_AUTO_DOWNLOAD
            )

demo.launch(server_name='0.0.0.0', share=True, theme=THEME, css=CSS)
